In [27]:
import geopandas as gpd
import pandas as pd
import folium
import mapclassify
from jinja2 import Template
from branca.element import Element

# Variables
Location = '/home/martin/python_projects/Data_www/'
F1csv = 'Elections2026_top3.csv'
F2json = 'Sheffield_Wards.geojson'

# Load datasets
gdf = gpd.read_file(Location + F2json)
df = pd.read_csv(Location + F1csv)

# Fix GeoJSON - strip " Ward" suffix
gdf['Ward'] = gdf['admin_name'].str.replace(' Ward', '', regex=False).str.strip()

# Fix CSV - clean whitespace including non-breaking spaces
df['Ward'] = df['Ward'].str.replace('\xa0', ' ', regex=False).str.strip()

# Normalise party name variants
party_map = {
    'Labour and Co-operative Party': 'Labour Party',
    'ReformUK - Changing Politics for Good': 'Reform UK'
}
df['Description'] = df['Description'].replace(party_map)

# Assign colours
party_colours = {
    'Green Party':       '#4CAF50',
    'Liberal Democrats': '#FAA61A',
    'Labour Party':      '#E4003B',
    'Reform UK':         '#12B6CF',
    'Independent':       '#888888'
}

# Get winner per ward
winners = df.loc[df.groupby('Ward')['Votes'].idxmax()].reset_index(drop=True)
winners = winners.drop(columns=['Name'])

# Merge with GeoJSON
gdf_merged = gdf.merge(winners[['Ward', 'Description', 'Votes']], on='Ward')
gdf_merged['colour'] = gdf_merged['Description'].map(party_colours)

# Build ward results dict
ward_results = {}
for ward, group in df.groupby('Ward'):
    group_sorted = group.sort_values('Votes', ascending=False).reset_index(drop=True)
    ward_results[ward] = group_sorted[['Description', 'Votes']].to_dict('records')

# Special cases
special_notes = {
    'Firth Park': '⚠️ Two seats contested — Reform UK & Labour Party each won one seat.'
}

# Base map
m = folium.Map(location=[53.3811, -1.4701], zoom_start=11, tiles='CartoDB positron')

# Inject CSS for tooltip sizing
meta = Element("""
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<style>
    .leaflet-tooltip {
        font-size: 14px !important;
        min-width: 260px !important;
        padding: 10px !important;
        pointer-events: none;
    }
</style>
""")
m.get_root().header.add_child(meta)

# Add wards
for _, row in gdf_merged.iterrows():
    ward = row['Ward']
    results = ward_results[ward]
    max_votes = max(r['Votes'] for r in results)

    bars_html = ""
    for r in results:
        colour = party_colours.get(r['Description'], '#888888')
        bar_width = int((r['Votes'] / max_votes) * 200)
        bars_html += f"""
        <div style="margin: 6px 0; font-size: 13px;">
            <div style="margin-bottom:2px;">{r['Description']}: <b>{r['Votes']}</b></div>
            <div style="background:{colour}; width:{bar_width}px; height:14px; border-radius:3px;"></div>
        </div>
        """

    note = special_notes.get(ward, '')
    note_html = f'<div style="font-size:11px; color:#888; margin-top:8px;">{note}</div>' if note else ''

    tooltip_html = f"""
    <div style="font-family:Arial; min-width:250px; padding:8px;">
        <b style="font-size:15px;">{ward}</b><br><br>
        {bars_html}
        {note_html}
    </div>
    """

    folium.GeoJson(
        row['geometry'],
        style_function=lambda x, colour=row['colour']: {
            'fillColor': colour,
            'color': 'white',
            'weight': 1.5,
            'fillOpacity': 0.5
        },
        tooltip=folium.Tooltip(tooltip_html, sticky=True)
    ).add_to(m)

# Legend
legend_html = """
<div style="position: fixed; bottom: 40px; left: 40px; z-index: 1000;
            background-color: white; padding: 20px; border-radius: 8px;
            box-shadow: 2px 2px 6px rgba(0,0,0,0.3); font-family: Arial; font-size: 15px;">
    <b style="font-size:16px;">Sheffield 2026 — Ward Winners</b><br><br>
    {% for party, colour in parties.items() %}
    <div style="margin: 6px 0;">
        <span style="background:{{ colour }}; width:18px; height:18px;
                     display:inline-block; border-radius:3px; margin-right:10px;"></span>
        {{ party }}
    </div>
    {% endfor %}
    <br>
    <div style="font-size:12px; color:#555; border-top: 1px solid #ddd; padding-top:10px;">
        Data: Sheffield City Council, 2026<br>
        Built by encephalon_zest<br>
        Co-authored with Claude (Anthropic)<br>
        <i>Shows 2026 sentiment per ward — one seat per ward.<br>
        Not full council composition.</i>
    </div>
</div>
"""

legend = Template(legend_html).render(parties=party_colours)
m.get_root().html.add_child(folium.Element(legend))

m.save(Location + 'index.html')
print("Done!")

Done!
